In [15]:
import pandas as pd
import importlib
utils = importlib.import_module("eval-app.codes.utils")

# Toxicity type prompts

### Old code (similarity threshold)

In [ ]:
# import copy
# import yaml

# class MyDumper(yaml.Dumper):

#     def increase_indent(self, flow=False, indentless=False):
#         return super(MyDumper, self).increase_indent(flow, indentless)

# test_template = {
#     'description': 'Test similarity',
#     'vars': {
#         'chemname': 'Aspirin',
#         'casrn': '50-78-2',
#         'effect_type': 'Sub acute toxicity'
#     },
#     'assert': []
# }

# test_assert_template = {
#         'type': 'assert-set',
#         'threshold': 0.4,
#         'assert':[{
#                 'type': 'similar',
#                 'value': 'Gastrointestinal bleeding',
#                 'threshold': 0.8
#             },
#             {
#                 'type': 'contains',
#                 'value': 'Gastrointestinal bleeding'
#             }
#         ]
#     }

# def createTests(info, species):

#     tests = []

#     for i, row in info.iterrows():
#         for col in info.columns:
#             if col in ['Other', 'Toxicity']: continue
#             if row[col]:
#                 test_temp = copy.deepcopy(test_template)
#                 chemname, casrn = col.split(', ')
#                 test_temp['vars']['chemname'] = chemname
#                 test_temp['vars']['casrn'] = casrn
#                 if row['Toxicity']:
#                     test_temp['vars']['effect_type'] = f"{row['Toxicity']} toxicity"
#                 else:
#                     test_temp['vars']['effect_type'] = row['Other']
#                 for term in str(row[col]).split('; '):
#                     test_assert_temp = copy.deepcopy(test_assert_template)
#                     test_assert_temp['assert'][0]['value'] = term
#                     test_assert_temp['assert'][1]['value'] = term
#                     test_temp['assert'].append(test_assert_temp)
#                 tests.append(test_temp)

#     with open(utils.Config.DIR_HOME / 'tests' / f'tox-type-assertion-prompts-{species}-prompts.yaml', 'w') as outfile:
#         prompts = [f"\'Answer the following question with a list of string. List the {{effect_type}} of {{chemname}} or its synonyms with CAS number {{casrn}} on {species}\'"]
#         yaml.dump(prompts, outfile, Dumper=MyDumper, default_flow_style=False)

#     with open(utils.Config.DIR_HOME / 'tests' / f'tox-type-assertion-prompts-{species}-tests.yaml', 'w') as outfile:
#         yaml.dump(tests, outfile, Dumper=MyDumper, default_flow_style=False)


### Check similarity using a python script

In [37]:
import copy
import yaml

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, indentless)

test_template_no_assertion = {
    'description': 'Test similarity',
    'vars': {
        'chemname': 'Aspirin',
        'casrn': '50-78-2',
        'effect_type': 'Sub acute toxicity'
    }
}

test_template_assertion = {
    'description': 'Test similarity',
    'vars': {
        'chemname': 'Aspirin',
        'casrn': '50-78-2',
        'effect_type': 'Sub acute toxicity'
    },
    'assert': [
        {
            'type': 'python', 
            'value': 'file://scripts/tests.py',
            'expected_phrases': ''
        }
    ]
}

def createTests(info, species):

    tests = []

    for i, row in info.iterrows():
        for col in info.columns:
            if col in ['Other', 'Toxicity']: continue
            test_temp = copy.deepcopy(test_template_assertion if row[col] else test_template_no_assertion)
            chemname, casrn = col.split(', ')
            test_temp['vars']['chemname'] = chemname
            test_temp['vars']['casrn'] = casrn
            if row['Toxicity']:
                test_temp['vars']['effect_type'] = f"{row['Toxicity']} toxicity"
            else:
                test_temp['vars']['effect_type'] = row['Other']
            if row[col]:
                test_temp['assert'][0]['expected_phrases'] = str(row[col]).split('; ')
            tests.append(test_temp)

    with open(utils.Config.DIR_HOME / 'tests' / f'tox-type-assertion-prompts-{species}-prompts.yaml', 'w') as outfile:
        prompts = ["\'Answer the following question with a list of string. List the {{effect_type}} of {{chemname}} or its synonyms with CAS number {{casrn}} on " + species + "\'"]
        yaml.dump(prompts, outfile, Dumper=MyDumper, default_flow_style=False)

    with open(utils.Config.DIR_HOME / 'tests' / f'tox-type-assertion-prompts-{species}-tests.yaml', 'w') as outfile:
        yaml.dump(tests, outfile, Dumper=MyDumper, default_flow_style=False)


### Human

In [38]:
info = pd.read_csv(utils.Config.DIR_HOME / 'docs' / 'chemical_effect_human.txt', delimiter='\t', dtype='str')
info[info.isna()] = ''
info.head()

,Other,Toxicity,"Aspirin, 50-78-2","Paracetamol, 103-90-2","Metformin, 657-24-9","Atorvastatin, 134523-00-5","Prednisone, 53-03-2","Dioxin, 1746-01-6","Benzene, 71-43-2","Arsenite, 15502-74-6",...,"6PPQ (N-(1,3-dimethylbutyl)-N'-phenyl-p-phenylenediamine), 2754428-18-5","Phenanthrene, 85-01-8","TPHP (triphenylphosphate), 115-86-6","Lanthanum, 7439-91-0","Galaxolide, 1222-05-5","1,6-Dimethyl-3,4-dihydroisoquinoline, 91753-09-2","2-Methyl-4-(4-methylphenyl)-2,3-dihydro-1,5-benzothiazepine, 74148-62-2","N-Cyclopropylmethyl-10,11-dihydro-5H-dibenzo-(a,d)-cyclohepten-5-imine, 59864-46-9","4-Methyl-1,2-dihydrobenzo[f]isoquinoline, 29248-42-8","6-Methyl-2,5-diphenyl-6H-1,3,4-thiadiazine, 62625-70-1"
0,,Sub acute,Gastrointestinal bleeding,Hypothermia; Acidosis; Hepatocellular/liver in...,Acidosis,,Weight gain; Increased appetite,Altered liver function,,Fatty degeneration of the heart,...,,,,,,,,,,
1,,Chronic,Gastrointestinal bleeding; Anemia; Chronic kid...,,,,Osteoporosis; Weight gain; Increased appetite;...,Breast cancer; Endometrial cancer; Testis cancer,Anemia; Bone marrow suppression; Immune suppre...,Anemia; Leukopenia; Liver damage; Hyperpigment...,...,,,,,,,,,,
2,,Developmental,Gastroschisis,,,,,,,,...,,,,,,,,,,
3,,Immune/lymphatic system,Hypersensitivity/allergic reaction,,,,Edema; Immune suppression,Immune suppression,,,...,,,,,,,,,,
4,,Reproductive system,,,,,,Endometriosis; Decreased sperm count,,,...,,,,,,,,,,


In [39]:
createTests(info, species='human')

### Rat

In [42]:
info = pd.read_csv(utils.Config.DIR_HOME / 'docs' / 'chemical_effect_rat.txt', delimiter='\t', dtype='str')
info[info.isna()] = ''
info.head()

,Other,Toxicity,"Aspirin, 50-78-2","Paracetamol, 103-90-2","Metformin, 657-24-9","Atorvastatin, 134523-00-5","Prednisone, 53-03-2","Dioxin, 1746-01-6","Benzene, 71-43-2","Arsenite, 15502-74-6",...,"6PPQ (N-(1,3-dimethylbutyl)-N'-phenyl-p-phenylenediamine), 2754428-18-5","Phenanthrene, 85-01-8","TPHP (triphenylphosphate), 115-86-6","Lanthanum, 7439-91-0","Galaxolide, 1222-05-5","1,6-Dimethyl-3,4-dihydroisoquinoline, 91753-09-2","2-Methyl-4-(4-methylphenyl)-2,3-dihydro-1,5-benzothiazepine, 74148-62-2","N-Cyclopropylmethyl-10,11-dihydro-5H-dibenzo-(a,d)-cyclohepten-5-imine, 59864-46-9","4-Methyl-1,2-dihydrobenzo[f]isoquinoline, 29248-42-8","6-Methyl-2,5-diphenyl-6H-1,3,4-thiadiazine, 62625-70-1"
0,,Sub acute,Gastrointestinal bleeding,,,,,,,,...,,,,,,,,,,
1,,Chronic,Angiosarcoma of the liver,,,,,,,,...,,,,,,,,,,
2,,Developmental,,,,,,,,,...,,,,,,,,,,
3,,Immune/lymphatic system,,,,,,,,,...,,,,,,,,,,
4,,Reproductive system,,,,,,,,,...,,,,,,,,,,


In [43]:
createTests(info, species='rat')

# ABT QA

In [4]:
info = pd.read_json(utils.Config.DIR_HOME / 'docs' / "abt_qa_wo_noise.json")
info[info.isna()] = ''
info

,Question Index,Questions,Options,Question Source,Answers,Answer Source
0,2000A Q1,Which of the following pairs of radionuclides ...,"{'A': '222 Rn and 137 Cs', 'B': '222 Rn and 90...",ABT2000#17.pdf,[[C]],"ABT #17.pdf, Document.pdf"
1,2000A Q2,A semi-conscious pesticide sprayer is brought ...,{'A': 'propanolol followed by an intravenous a...,ABT2000#17.pdf,[[D]],"ABT #17.pdf, Document.pdf"
2,2000A Q3,Botulinum toxin inhibits neuromuscular communi...,{'A': 'blocking sodium channels along the moto...,ABT2000#17.pdf,[[B]],"ABT #17.pdf, Document.pdf"
3,2000A Q5,Contact urticaria is characteristic of,"{'A': 'mistletoe', 'B': 'crocus', 'C': 'pokewe...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf",[[D]],"ABT #17.pdf, Document.pdf"
4,2000A Q6,The dominant route of excretion for absorbed l...,"{'A': 'bile', 'B': 'feces', 'C': 'urine', 'D':...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf",[[C]],"ABT #17.pdf, Document.pdf"
...,...,...,...,...,...,...
469,2007C Q36,"The following factors, identified by Bradford-...","{'A': 'consistency of the association', 'B': '...",24-2007-recert no answers.pdf,[[E]],ABT Recert 24-2007 MH group Answers.doc
470,2007C Q37,The probability that a chemical may produce ca...,{'A': 'whether comparable peak plasma concentr...,24-2007-recert no answers.pdf,"[[A, E]]",ABT Recert 24-2007 MH group Answers.doc
471,2007C Q38,Elevation of which of the following serum/plas...,"{'A': 'troponin I', 'B': 'alkaline phosphatase...",24-2007-recert no answers.pdf,[[C]],ABT Recert 24-2007 MH group Answers.doc
472,2007C Q39,Short Term Exposure Limits (STELs) are,{'A': 'the absolute maximum concentrations to ...,24-2007-recert no answers.pdf,[[C]],ABT Recert 24-2007 MH group Answers.doc


In [14]:
import copy
import yaml

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, indentless)

test_template = {
    'description': 'Test similarity',
    'vars': {
        'question': '',
        'options': ''
    },
    'assert': [
        {
            'type': 'python', 
            'value': 'file://scripts/tests.py',
            'expected_phrases': ''
        }
    ]
}

tests = []

answers_noisy = []
for i, row in info.iterrows():
    if len(row['Answers']) > 1: 
        continue
    try:
        test_temp = copy.deepcopy(test_template)
        test_temp['vars']['question'] = row['Questions']
        test_temp['vars']['options'] = '\n\n'.join([f'({k}) {v}' for k, v in row['Options'].items()])
        test_temp['assert'][0]['expected_phrases'] = list(map(lambda x: f'({x.strip()}) {row['Options'][x.strip()]}', row['Answers'][0]))
    except Exception as exp:
        answers_noisy.append(row)
        continue
    tests.append(test_temp)

with open(utils.Config.DIR_HOME / 'tests' / f'abt-assertion-prompts.yaml', 'w') as outfile:
    prompts = ["Consider the following question followed by options. Answer the question from choosing the options. You can choose multiple options. Each option must start with the option number.\n\n**Question**:\n\n{{question}}\n\n**Options**:\n\n{{options}}\n"]
    yaml.dump(prompts, outfile, Dumper=MyDumper, default_flow_style=False)

with open(utils.Config.DIR_HOME / 'tests' / 'abt-assertion-prompts-tests.yaml', 'w') as outfile:
    yaml.dump(tests, outfile, Dumper=MyDumper, default_flow_style=False)

pd.DataFrame(answers_noisy)

""
